# Phase 5 Bayesian Tuning - Label 1 vs Label 2

**Strategy:** Phases 1–4 are fixed (best config already found). All tuning budget goes to Phase 5.

- **Exploration:** 10-fold stratified CV on ~240 Label 1+2 samples, Optuna TPE, 2000 trials
- **Validation:** LOOCV on top 10 configs from Optuna -> final top 5 ranking
- **Resume:** stop anytime, re-run to continue. Graphs and top 5 work from checkpoint.


In [1]:
import os, random, warnings, json, time
from datetime import timedelta
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix
import optuna
from optuna.samplers import TPESampler
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
SEED = 359956
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)



c:\Users\simop\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
for res in ["punkt", "punkt_tab", "wordnet", "stopwords",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(res, quiet=True)
df = pd.read_csv("train.csv").dropna(subset=["TEXT", "LABEL"]).reset_index(drop=True)
print(f"Loaded {len(df)} rows")
lemmatizer = WordNetLemmatizer()
def clean_text_none(text):
    text = str(text).lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha()]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)
print("Cleaning texts ...")
df["text_clean"] = df["TEXT"].apply(clean_text_none)
print("Extracting stylometric features ...")
def extract_stylometric(text):
    text = str(text)
    sentences = sent_tokenize(text)
    n_sentences = max(len(sentences), 1)
    sent_lengths = [len(word_tokenize(s)) for s in sentences]
    words = word_tokenize(text)
    alpha_words = [w for w in words if w.isalpha()]
    n_words = max(len(alpha_words), 1)
    word_lengths = [len(w) for w in alpha_words] if alpha_words else [0]
    n_chars = max(len(text), 1)
    n_commas = text.count(",")
    n_periods = text.count(".")
    n_exclaim = text.count("!")
    n_question = text.count("?")
    n_semicolon = text.count(";")
    n_colon = text.count(":")
    n_dash = text.count("\u2014") + text.count("-")
    n_parens = text.count("(") + text.count(")")
    n_quotes = text.count('"') + text.count("'") + text.count("\u201c") + text.count("\u201d")
    try:
        tags = pos_tag(alpha_words[:200])
        tag_counts = Counter(t for _, t in tags)
        n_tagged = max(sum(tag_counts.values()), 1)
        pct_noun = (tag_counts.get("NN", 0) + tag_counts.get("NNS", 0) + tag_counts.get("NNP", 0) + tag_counts.get("NNPS", 0)) / n_tagged
        pct_verb = (tag_counts.get("VB", 0) + tag_counts.get("VBD", 0) + tag_counts.get("VBG", 0) + tag_counts.get("VBN", 0) + tag_counts.get("VBP", 0) + tag_counts.get("VBZ", 0)) / n_tagged
        pct_adj = (tag_counts.get("JJ", 0) + tag_counts.get("JJR", 0) + tag_counts.get("JJS", 0)) / n_tagged
        pct_adv = (tag_counts.get("RB", 0) + tag_counts.get("RBR", 0) + tag_counts.get("RBS", 0)) / n_tagged
    except:
        pct_noun = pct_verb = pct_adj = pct_adv = 0.0
    unique_words = set(w.lower() for w in alpha_words)
    ttr = len(unique_words) / n_words
    pct_short = sum(1 for w in alpha_words if len(w) <= 3) / n_words
    pct_long = sum(1 for w in alpha_words if len(w) >= 8) / n_words
    return {"n_chars": n_chars, "n_words": n_words, "n_sentences": n_sentences,
        "avg_word_len": np.mean(word_lengths), "std_word_len": np.std(word_lengths),
        "avg_sent_len": np.mean(sent_lengths),
        "std_sent_len": np.std(sent_lengths) if len(sent_lengths) > 1 else 0,
        "max_sent_len": max(sent_lengths), "min_sent_len": min(sent_lengths),
        "comma_rate": n_commas / n_words, "period_rate": n_periods / n_words,
        "exclaim_rate": n_exclaim / n_words, "question_rate": n_question / n_words,
        "semicolon_rate": n_semicolon / n_words, "colon_rate": n_colon / n_words,
        "dash_rate": n_dash / n_words, "paren_rate": n_parens / n_words,
        "quote_rate": n_quotes / n_words,
        "total_punct_rate": (n_commas + n_periods + n_exclaim + n_question + n_semicolon + n_colon) / n_words,
        "pct_noun": pct_noun, "pct_verb": pct_verb, "pct_adj": pct_adj, "pct_adv": pct_adv,
        "noun_verb_ratio": pct_noun / max(pct_verb, 0.001),
        "ttr": ttr, "pct_short_words": pct_short, "pct_long_words": pct_long,
        "words_per_sentence": n_words / n_sentences, "chars_per_word": n_chars / n_words}
stylo_df = pd.DataFrame(df["TEXT"].apply(extract_stylometric).tolist())
STYLO_ALL = stylo_df.values.astype(float)
STYLO_GROUPS = {
    "all_30": list(range(29)), "punctuation": list(range(9, 19)),
    "pos": list(range(19, 24)), "length": list(range(0, 9)),
    "vocabulary": list(range(24, 27)), "structure": [27, 28],
    "punct_pos": list(range(9, 24)),
    "punct_pos_vocab": list(range(9, 24)) + list(range(24, 27)),
    "no_length": list(range(9, 29)),
    "no_pos": list(range(0, 19)) + list(range(24, 29)),
}
print(f"Stylometric features: {STYLO_ALL.shape}")
print("Preprocessing done.\n")



Loaded 2400 rows
Cleaning texts ...
Extracting stylometric features ...
Stylometric features: (2400, 29)
Preprocessing done.



In [ ]:
MASK_12 = np.isin(df["LABEL"].values, [1, 2])
IDX_12 = np.where(MASK_12)[0]
TEXTS_12 = df["text_clean"].values[IDX_12]
LABELS_12 = df["LABEL"].values[IDX_12]
STYLO_12 = STYLO_ALL[IDX_12]
print(f"Label 1+2 subset: {len(IDX_12)} samples")
print(f"  Label 1: {(LABELS_12 == 1).sum()}, Label 2: {(LABELS_12 == 2).sum()}")
N_FOLDS = 10
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLD_INDICES = list(skf.split(TEXTS_12, LABELS_12))
N_TRIALS = 3740
TUNING_DIR = Path("phase5_tuning")
TUNING_DIR.mkdir(exist_ok=True)
CHECKPOINT_FILE = TUNING_DIR / "checkpoint.json"
RESULTS_FILE = TUNING_DIR / "results.json"
TOP5_FILE = TUNING_DIR / "top5.json"
TOP5_LOOCV_FILE = TUNING_DIR / "top5_loocv.json"
DB_FILE = TUNING_DIR / "optuna_study.db"



Label 1+2 subset: 240 samples
  Label 1: 80, Label 2: 160


In [4]:
def build_impchi(X_tr_text, X_val_text, y_tr, k_per_class, max_feat=15000, ngram_max=2, min_df=2, sublinear=True):
    tfidf = TfidfVectorizer(max_features=max_feat, ngram_range=(1, ngram_max), sublinear_tf=sublinear, min_df=min_df, strip_accents="unicode", analyzer="word")
    X_tr_full = tfidf.fit_transform(X_tr_text)
    X_val_full = tfidf.transform(X_val_text)
    classes = np.unique(y_tr)
    selected = set()
    for cls in classes:
        y_bin = (y_tr == cls).astype(int)
        scores, _ = chi2(X_tr_full, y_bin)
        top_k = np.argsort(scores)[-k_per_class:]
        selected.update(top_k)
    sel = sorted(selected)
    return X_tr_full[:, sel], X_val_full[:, sel]

def build_stylo(tr_idx, val_idx, col_indices):
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(STYLO_12[tr_idx][:, col_indices])
    X_val = scaler.transform(STYLO_12[val_idx][:, col_indices])
    return csr_matrix(X_tr), csr_matrix(X_val)



In [5]:
def objective(trial):
    impchi_k = trial.suggest_int("impchi_k", 10, 100)
    impchi_max_feat = trial.suggest_int("impchi_max_feat", 5000, 20000, log=True)
    impchi_ngram = trial.suggest_int("impchi_ngram_max", 1, 3)
    impchi_min_df = trial.suggest_int("impchi_min_df", 1, 5)
    impchi_sublinear = trial.suggest_categorical("impchi_sublinear", [True, False])
    stylo_subset = trial.suggest_categorical("stylo_subset",
        ["all_30", "punctuation", "pos", "length", "vocabulary",
         "punct_pos", "punct_pos_vocab", "no_length", "no_pos"])
    stylo_cols = STYLO_GROUPS[stylo_subset]
    hgb_max_iter = trial.suggest_int("hgb_max_iter", 30, 300)
    hgb_max_depth = trial.suggest_int("hgb_max_depth", 2, 16)
    hgb_lr = trial.suggest_float("hgb_lr", 0.005, 0.3, log=True)
    hgb_leaves = trial.suggest_int("hgb_max_leaf_nodes", 3, 31)
    hgb_min_leaf = trial.suggest_int("hgb_min_samples_leaf", 2, 30)
    hgb_l2 = trial.suggest_float("hgb_l2", 0.0, 10.0)
    hgb_cw = trial.suggest_categorical("hgb_class_weight", ["balanced", "none"])
    fold_scores = []
    for tr_idx, val_idx in FOLD_INDICES:
        y_tr = LABELS_12[tr_idx]
        y_val = LABELS_12[val_idx]
        X_tr_chi, X_val_chi = build_impchi(TEXTS_12[tr_idx], TEXTS_12[val_idx], y_tr, k_per_class=impchi_k, max_feat=impchi_max_feat, ngram_max=impchi_ngram, min_df=impchi_min_df, sublinear=impchi_sublinear)
        X_tr_sty, X_val_sty = build_stylo(tr_idx, val_idx, stylo_cols)
        X_tr = hstack([X_tr_chi, X_tr_sty]).toarray()
        X_val = hstack([X_val_chi, X_val_sty]).toarray()
        m = HistGradientBoostingClassifier(max_iter=hgb_max_iter, max_depth=hgb_max_depth, learning_rate=hgb_lr, max_leaf_nodes=hgb_leaves, min_samples_leaf=hgb_min_leaf, l2_regularization=hgb_l2, class_weight="balanced" if hgb_cw == "balanced" else None, random_state=SEED)
        m.fit(X_tr, y_tr)
        preds = m.predict(X_val)
        fold_scores.append(f1_score(y_val, preds, average="macro"))
    return float(np.mean(fold_scores))



In [6]:
def save_checkpoint(study):
    records = []
    for t in study.trials:
        if t.state == optuna.trial.TrialState.COMPLETE:
            records.append({"number": t.number, "f1_mean": t.value, "params": t.params, "duration_s": round(t.duration.total_seconds(), 1) if t.duration else None})
    records.sort(key=lambda x: x["f1_mean"], reverse=True)
    with open(CHECKPOINT_FILE, "w") as fp:
        json.dump(records, fp, indent=2, default=str)
    return len(records)

def load_completed_count():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, "r") as fp:
            return len(json.load(fp))
    return 0



In [7]:
already_done = load_completed_count()
remaining = max(0, N_TRIALS - already_done)
print(f"Target: {N_TRIALS} trials")
print(f"Already completed: {already_done}")
print(f"Remaining: {remaining}")
if remaining == 0:
    print("All trials completed! Skipping to results.")
else:
    storage_path = f"sqlite:///{DB_FILE}"
    study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=SEED), study_name="phase5_final", storage=storage_path, load_if_exists=True)
    t0 = time.time()
    best_so_far = [0.0]
    completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    if completed:
        best_so_far[0] = max(t.value for t in completed)
    def trial_callback(study, trial):
        elapsed = time.time() - t0
        n_new = trial.number + 1 - already_done
        if n_new <= 0:
            return
        avg = elapsed / n_new
        rem_time = avg * (remaining - n_new)
        if trial.value and trial.value > best_so_far[0]:
            best_so_far[0] = trial.value
        if n_new % 10 == 0 or n_new == 1:
            print(f"  Trial {trial.number+1:>5d} (new #{n_new}/{remaining})  F1={trial.value:.4f}  best={best_so_far[0]:.4f}  | {timedelta(seconds=int(elapsed))} elapsed  ~{timedelta(seconds=int(rem_time))} remaining")
        if n_new % 10 == 0:
            n_saved = save_checkpoint(study)
            print(f"    [CHECKPOINT] {n_saved} trials -> {CHECKPOINT_FILE}")
    print(f"\nStarting Optuna ({remaining} trials) ...\n")
    study.optimize(objective, n_trials=remaining, callbacks=[trial_callback])
    total_time = time.time() - t0
    print(f"\nFinished! {remaining} new trials in {timedelta(seconds=int(total_time))}")
    print(f"Best F1-macro: {study.best_value:.4f}")
    save_checkpoint(study)



Target: 1000000 trials
Already completed: 1669
Remaining: 998331

Starting Optuna (998331 trials) ...

  Trial  1679 (new #10/998331)  F1=0.8643  best=0.8925  | 0:00:11 elapsed  ~13 days, 5:56:27 remaining
    [CHECKPOINT] 1677 trials -> phase5_tuning\checkpoint.json
  Trial  1689 (new #20/998331)  F1=0.8530  best=0.8925  | 0:00:49 elapsed  ~28 days, 15:56:48 remaining
    [CHECKPOINT] 1687 trials -> phase5_tuning\checkpoint.json
  Trial  1699 (new #30/998331)  F1=0.8360  best=0.8925  | 0:01:29 elapsed  ~34 days, 8:52:40 remaining
    [CHECKPOINT] 1697 trials -> phase5_tuning\checkpoint.json
  Trial  1709 (new #40/998331)  F1=0.8688  best=0.8925  | 0:02:10 elapsed  ~37 days, 16:18:47 remaining
    [CHECKPOINT] 1707 trials -> phase5_tuning\checkpoint.json
  Trial  1719 (new #50/998331)  F1=0.8755  best=0.8925  | 0:02:50 elapsed  ~39 days, 11:21:58 remaining
    [CHECKPOINT] 1717 trials -> phase5_tuning\checkpoint.json
  Trial  1729 (new #60/998331)  F1=0.8490  best=0.8925  | 0:03:28 ela

[W 2026-03-17 16:01:43,404] Trial 3741 failed with parameters: {'impchi_k': 36, 'impchi_max_feat': 8655, 'impchi_ngram_max': 1, 'impchi_min_df': 2, 'impchi_sublinear': True, 'stylo_subset': 'no_pos', 'hgb_max_iter': 141, 'hgb_max_depth': 14, 'hgb_lr': 0.06849622784819132, 'hgb_max_leaf_nodes': 16, 'hgb_min_samples_leaf': 2, 'hgb_l2': 6.490032253439827, 'hgb_class_weight': 'balanced'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\simop\AppData\Local\Programs\Python\Python311\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\simop\AppData\Local\Temp\ipykernel_2220\815753070.py", line 27, in objective
    m.fit(X_tr, y_tr)
  File "c:\Users\simop\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^

KeyboardInterrupt: 

In [ ]:
if CHECKPOINT_FILE.exists():
    with open(CHECKPOINT_FILE, "r") as fp:
        all_results = json.load(fp)
    with open(RESULTS_FILE, "w") as fp:
        json.dump(all_results, fp, indent=2, default=str)
    print(f"Saved {len(all_results)} trials -> {RESULTS_FILE}")
else:
    all_results = []
    print("No checkpoint found.")



In [ ]:
if not all_results and CHECKPOINT_FILE.exists():
    with open(CHECKPOINT_FILE, "r") as fp:
        all_results = json.load(fp)
print(f"\nLoaded {len(all_results)} trials from checkpoint.\n")
print("=" * 90)
print("  TOP 10 FROM OPTUNA (by 10-fold macro F1)")
print("=" * 90)
top10_optuna = all_results[:10]
for rank, r in enumerate(top10_optuna, 1):
    p = r["params"]
    print(f"\n  #{rank}  Trial {r['number']}  F1_10fold = {r['f1_mean']:.4f}")
    print(f"    ImpCHI: k={p['impchi_k']}, max_feat={p['impchi_max_feat']}, ngram=(1,{p['impchi_ngram_max']}), min_df={p['impchi_min_df']}, sublinear={p['impchi_sublinear']}")
    print(f"    Stylo:  {p['stylo_subset']}")
    print(f"    HistGB: max_iter={p['hgb_max_iter']}, depth={p['hgb_max_depth']}, lr={p['hgb_lr']:.4f}, leaves={p['hgb_max_leaf_nodes']}, min_leaf={p['hgb_min_samples_leaf']}, l2={p['hgb_l2']:.2f}, cw={p['hgb_class_weight']}")



In [ ]:
print("\n" + "=" * 90)
print("  LOOCV VALIDATION - Top 10 configs from Optuna")
print("=" * 90)
print(f"  Running LOOCV on {len(LABELS_12)} samples for each config ...\n")
loocv_results = []
n_samples = len(LABELS_12)
for rank, r in enumerate(top10_optuna, 1):
    p = r["params"]
    stylo_cols = STYLO_GROUPS[p["stylo_subset"]]
    loo_preds = np.zeros(n_samples, dtype=int)
    t_start = time.time()
    for i in range(n_samples):
        tr_mask = np.ones(n_samples, dtype=bool)
        tr_mask[i] = False
        tr_idx = np.where(tr_mask)[0]
        val_idx = np.array([i])
        y_tr = LABELS_12[tr_idx]
        X_tr_chi, X_val_chi = build_impchi(TEXTS_12[tr_idx], TEXTS_12[val_idx], y_tr, k_per_class=p["impchi_k"], max_feat=p["impchi_max_feat"], ngram_max=p["impchi_ngram_max"], min_df=p["impchi_min_df"], sublinear=p["impchi_sublinear"])
        X_tr_sty, X_val_sty = build_stylo(tr_idx, val_idx, stylo_cols)
        X_tr = hstack([X_tr_chi, X_tr_sty]).toarray()
        X_val = hstack([X_val_chi, X_val_sty]).toarray()
        m = HistGradientBoostingClassifier(max_iter=p["hgb_max_iter"], max_depth=p["hgb_max_depth"], learning_rate=p["hgb_lr"], max_leaf_nodes=p["hgb_max_leaf_nodes"], min_samples_leaf=p["hgb_min_samples_leaf"], l2_regularization=p["hgb_l2"], class_weight="balanced" if p["hgb_class_weight"] == "balanced" else None, random_state=SEED)
        m.fit(X_tr, y_tr)
        loo_preds[i] = m.predict(X_val)[0]
    f1_loo = f1_score(LABELS_12, loo_preds, average="macro")
    elapsed = time.time() - t_start
    loocv_results.append({"optuna_rank": rank, "trial": r["number"], "f1_10fold": r["f1_mean"], "f1_loocv": f1_loo, "params": p, "loocv_time_s": round(elapsed, 1)})
    print(f"  #{rank:>2d}  Trial {r['number']:>5d}  F1_10fold={r['f1_mean']:.4f}  ->  F1_LOOCV={f1_loo:.4f}  ({elapsed:.0f}s)")



In [ ]:
loocv_results.sort(key=lambda x: x["f1_loocv"], reverse=True)
print("\n" + "=" * 90)
print("  FINAL TOP 5 (ranked by LOOCV macro F1)")
print("=" * 90)
top5_final = []
for rank, r in enumerate(loocv_results[:5], 1):
    p = r["params"]
    print(f"\n  -- #{rank}  Trial {r['trial']}  F1_LOOCV = {r['f1_loocv']:.4f}  (was #{r['optuna_rank']} in Optuna, F1_10fold={r['f1_10fold']:.4f}) --")
    print(f"    ImpCHI: k={p['impchi_k']}, max_feat={p['impchi_max_feat']}, ngram=(1,{p['impchi_ngram_max']}), min_df={p['impchi_min_df']}, sublinear={p['impchi_sublinear']}")
    print(f"    Stylo:  {p['stylo_subset']}")
    print(f"    HistGB: max_iter={p['hgb_max_iter']}, depth={p['hgb_max_depth']}, lr={p['hgb_lr']:.4f}, leaves={p['hgb_max_leaf_nodes']}, min_leaf={p['hgb_min_samples_leaf']}, l2={p['hgb_l2']:.2f}, cw={p['hgb_class_weight']}")
    top5_final.append({"rank": rank, "trial": r["trial"], "f1_loocv": r["f1_loocv"], "f1_10fold": r["f1_10fold"], "params": p})
with open(TOP5_LOOCV_FILE, "w") as fp:
    json.dump(top5_final, fp, indent=2, default=str)
print(f"\nSaved: {TOP5_LOOCV_FILE}")
with open(TOP5_FILE, "w") as fp:
    json.dump(loocv_results, fp, indent=2, default=str)
print(f"Saved: {TOP5_FILE}")



In [ ]:
if not all_results and CHECKPOINT_FILE.exists():
    with open(CHECKPOINT_FILE, "r") as fp:
        all_results = json.load(fp)
if all_results:
    trial_nums = [r["number"] for r in all_results]
    f1_vals = [r["f1_mean"] for r in all_results]
    order = np.argsort(trial_nums)
    t_sorted = np.array(trial_nums)[order]
    f1_sorted = np.array(f1_vals)[order]
    cummax = np.maximum.accumulate(f1_sorted)
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.scatter(t_sorted, f1_sorted, s=8, alpha=0.4, label="Trial F1")
    ax.plot(t_sorted, cummax, color="red", linewidth=1.5, label="Best so far")
    ax.set_xlabel("Trial")
    ax.set_ylabel("Phase 5 F1-macro (10-fold)")
    ax.set_title("Optimization History", fontweight="bold")
    ax.legend()
    plt.tight_layout()
    plt.show()

    df_r = pd.DataFrame(all_results)
    df_r["stylo"] = df_r["params"].apply(lambda p: p["stylo_subset"])
    df_r["hgb_cw"] = df_r["params"].apply(lambda p: p["hgb_class_weight"])
    df_r["impchi_k"] = df_r["params"].apply(lambda p: p["impchi_k"])
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    df_r.boxplot(column="f1_mean", by="stylo", ax=axes[0], rot=45)
    axes[0].set_title("F1 by Stylo Subset")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("F1-macro")
    df_r.boxplot(column="f1_mean", by="hgb_cw", ax=axes[1])
    axes[1].set_title("F1 by class_weight")
    axes[1].set_xlabel("")
    axes[2].scatter(df_r["impchi_k"], df_r["f1_mean"], s=8, alpha=0.3)
    axes[2].set_xlabel("impchi_k_per_class")
    axes[2].set_ylabel("F1-macro")
    axes[2].set_title("F1 vs ImpCHI K")
    plt.suptitle("")
    plt.tight_layout()
    plt.show()

    top50 = all_results[:50]
    param_keys = ["hgb_max_iter", "hgb_max_depth", "hgb_lr", "hgb_max_leaf_nodes", "hgb_min_samples_leaf", "hgb_l2", "impchi_k", "impchi_max_feat"]
    fig, axes = plt.subplots(2, 4, figsize=(16, 7))
    axes = axes.flatten()
    for idx, key in enumerate(param_keys):
        vals = [r["params"].get(key) for r in top50 if r["params"].get(key) is not None]
        if vals:
            axes[idx].hist(vals, bins=15, color="steelblue", edgecolor="white")
            axes[idx].set_title(key, fontsize=9)
    for j in range(len(param_keys), len(axes)):
        axes[j].set_visible(False)
    fig.suptitle("Top 50 Trials - Parameter Distributions", fontweight="bold")
    plt.tight_layout()
    plt.show()

    if loocv_results:
        fig, ax = plt.subplots(figsize=(6, 5))
        x = [r["f1_10fold"] for r in loocv_results]
        y = [r["f1_loocv"] for r in loocv_results]
        ax.scatter(x, y, s=60, zorder=3)
        for i, r in enumerate(loocv_results):
            ax.annotate(f"#{r['optuna_rank']}", (x[i], y[i]), textcoords="offset points", xytext=(5, 5), fontsize=8)
        lims = [min(min(x), min(y)) - 0.01, max(max(x), max(y)) + 0.01]
        ax.plot(lims, lims, "k--", alpha=0.3, label="y=x")
        ax.set_xlabel("10-fold F1")
        ax.set_ylabel("LOOCV F1")
        ax.set_title("10-fold vs LOOCV - Top 10 Configs", fontweight="bold")
        ax.legend()
        plt.tight_layout()
        plt.show()
else:
    print("No results to plot. Run the tuning first.")



In [ ]:
if all_results:
    df_a = pd.DataFrame([{"f1": r["f1_mean"], "stylo": r["params"]["stylo_subset"], "hgb_cw": r["params"]["hgb_class_weight"], "impchi_sublinear": r["params"]["impchi_sublinear"]} for r in all_results])
    print("\n" + "=" * 60)
    print("  AGGREGATED ANALYSIS")
    print("=" * 60)
    print("\n-- Per stylo subset --")
    print(df_a.groupby("stylo")["f1"].agg(["mean", "std", "max", "count"]).round(4).sort_values("max", ascending=False))
    print("\n-- Per class_weight --")
    print(df_a.groupby("hgb_cw")["f1"].agg(["mean", "std", "max", "count"]).round(4).sort_values("max", ascending=False))
    print("\n-- Per impchi_sublinear --")
    print(df_a.groupby("impchi_sublinear")["f1"].agg(["mean", "std", "max", "count"]).round(4).sort_values("max", ascending=False))
print("\n[DONE] Phase 5 tuning complete.")

